# 05_irisan — Langkah 14
Jalankan dari atas ke bawah. Sel 1 dan Sel 2 adalah SEL PEMBUKA.

## Sel 1 — SEL PEMBUKA (Python)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%load_ext rpy2.ipython
!apt-get -qq update > /dev/null
!apt-get -qq install -y libpoppler-cpp-dev libglpk-dev libxml2-dev libcurl4-openssl-dev libssl-dev libfontconfig1-dev libharfbuzz-dev libfribidi-dev libfreetype6-dev libpng-dev libtiff5-dev libjpeg-dev > /dev/null
print('Sel 1 selesai')

## Sel 2 — SEL PEMBUKA (R)

In [ ]:
%%R
proj <- "/content/drive/MyDrive/bibliometrik-cxr"
lib  <- file.path(proj, "Rlib")
dir.create(lib, recursive = TRUE, showWarnings = FALSE)
.libPaths(c(lib, .libPaths()))
codename <- system("lsb_release -cs", intern = TRUE)
options(repos = c(CRAN = sprintf("https://packagemanager.posit.co/cran/__linux__/%s/latest", codename)))
options(HTTPUserAgent = sprintf("R/%s R (%s)", getRversion(),
  paste(getRversion(), R.version$platform, R.version$arch, R.version$os)))
pkgs <- c("bibliometrix", "UpSetR", "readxl", "writexl", "ggplot2", "htmlwidgets")
baru <- pkgs[!pkgs %in% rownames(installed.packages())]
if (length(baru)) install.packages(baru, lib = lib)
setwd(proj)
suppressPackageStartupMessages(library(bibliometrix))
cat("Folder kerja:", getwd(), "\nbibliometrix", as.character(packageVersion("bibliometrix")), "siap\n")

## Sel 3 — UpSet plot irisan dan matriks ekstraksi L1 ∪ L2

In [ ]:
%%R
library(UpSetR); library(writexl)
M <- readRDS("03_clean/M.rds")
tg <- read.csv("03_clean/tugas.csv", check.names = FALSE)
norm <- function(x) gsub("[^a-z0-9]", "", tolower(x))
M$tugas <- tg$tugas[match(norm(M$TI), norm(tg$Title))]
cat("Dokumen tanpa pasangan tugas:", sum(is.na(M$tugas)), "\n")

txt <- tolower(paste(M$TI, M$AB, M$DE, M$ID))
f <- data.frame(
  MultiLabel = as.integer(grepl("multi-?label|multi label|single positive|partial label", txt)),
  Imbalance  = as.integer(grepl("imbalanc|long-?tail|long tail|rare disease|minority class", txt)),
  Shift      = as.integer(grepl(paste0("continual|lifelong|incremental learning|",
                  "domain shift|distribution shift|label shift|prior shift"), txt)))
print(colSums(f))
cat("L3 (multi-label dan imbalance):", sum(f$MultiLabel & f$Imbalance), "\n")
cat("L5 (imbalance dan shift):", sum(f$Imbalance & f$Shift), "\n")

png("05_figures/upset_irisan.png", width = 2400, height = 1500, res = 300)
print(upset(f, sets = c("MultiLabel", "Imbalance", "Shift"), order.by = "freq", keep.order = TRUE))
dev.off()

pilih <- f$MultiLabel == 1 | f$Imbalance == 1
mx <- data.frame(judul = M$TI[pilih], penulis = M$AU[pilih], tahun = M$PY[pilih],
                 sumber = M$SO[pilih], doi = M$DI[pilih], tugas = M$tugas[pilih],
                 L1_multilabel = f$MultiLabel[pilih], L2_imbalance = f$Imbalance[pilih],
                 L3_inti = as.integer(f$MultiLabel[pilih] & f$Imbalance[pilih]))
for (k in c("dataset", "rasio_label", "framework_ssl", "mekanisme_imbalance",
            "threshold_multilabel", "dependensi_label", "asumsi_prior",
            "shift", "metrik", "kode_tersedia", "catatan_kelemahan")) mx[[k]] <- ""
mx <- mx[order(-mx$L3_inti, mx$tugas != "klasifikasi", -mx$tahun), ]
write_xlsx(mx, "03_clean/matriks_ekstraksi.xlsx")
cat("Matriks ekstraksi:", nrow(mx), "dokumen (L3:", sum(mx$L3_inti), ")\n")
print(table(mx$tugas, useNA = "ifany"))